Generate independent candidate responses and select a consensus protocol under the configured call budget.


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/token_budget_controls/universal_self_consistency_deepseekv4_seven_samples'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import itertools
import json
import hashlib
import math
import random
import re
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Any, List
import numpy as np
import pandas as pd
import requests
from openai import OpenAI
from tqdm.auto import tqdm


In [ ]:
FOLDER_ID = ''
API_KEY = ''
LLM_API_URL = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'
INPUT_CSV = 'llmfmed_dataset.csv'
RAW_OUTPUT_CSV = 'usc_deepseekv4_raw_drafts.csv'
OUTPUT_CSV_BY_K = {3: 'usc_deepseekv4_sc3.csv', 5: 'usc_deepseekv4_sc5.csv', 7: 'usc_deepseekv4_sc7.csv'}
SC_MODEL_NAME = 'deepseekv4_default'
SC_MODEL_URI = f'gpt://{FOLDER_ID}/deepseek-v4-flash/latest'
CASE_COMPONENT_COLUMNS = ['Vignette', 'Review of Systems', 'Past Medical History', 'Physical Examination', 'Test Results']
EMERGENCY_LABELS = ['Self-care appropriate', 'Requires non-emergent care', 'Requires emergent care']
SUBCLINICAL_LABELS = ['Cardiology', 'Dermatology', 'Emergency', 'Endocrinology', 'Gastroenterology', 'General Medicine', 'General Surgery', 'Infectious Disease', 'Nephrology', 'Neurology', 'Obstetrics and Gynecology', 'Ophthalmology', 'Orthopedics', 'Other departments', 'Otolaryngology (ENT)', 'Pediatrics', 'Psychiatry', 'Pulmonology', 'Urology']
REQUEST_TIMEOUT = 120
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
K_SAMPLES = 7
K_LEVELS = [3, 5, 7]
TEMPERATURE_SC = 0.3
TEMPERATURE_USC = 0.3
MAX_TOKENS_PROTOCOL = 3000
MAX_TOKENS_USC = 2000
USC_DESIGN_BY_K = {3: 'orbit', 5: 'knockout', 7: 'knockout'}
MIN_PARSED_CALL_RATE = 0.5

def usc_call_plan(k: int) -> int:
    design = USC_DESIGN_BY_K[k]
    if design == 'orbit':
        return math.factorial(k)
    if design == 'knockout':
        return 2 * (k - 1)
    raise ValueError(f'Unknown design: {design}')
DELPHI_TOKENS_PER_CASE = 23567
DRAFT_COLUMNS = [f'sc_draft_{i}' for i in range(1, K_SAMPLES + 1)]
CASE_CONCURRENCY = 5
MAX_INFLIGHT_CALLS = 8
SC_ROLE_NAME = 'Default'
SC_ROLE_DESCRIPTION = 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'

def detect_provider(model_uri: str) -> str:
    if 'aliceai-llm' in model_uri:
        return 'yandex_rest'
    return 'openai_compatible'
SC_PROVIDER = detect_provider(SC_MODEL_URI)
df = pd.read_csv(INPUT_CSV, sep=None, engine='python')
df = df.copy().reset_index(drop=True)
missing_case_columns = [col for col in CASE_COMPONENT_COLUMNS if col not in df.columns]
if missing_case_columns:
    raise ValueError(f'Missing case columns in dataset: {missing_case_columns}')
if max(K_LEVELS) > K_SAMPLES:
    raise ValueError('K_LEVELS не может превышать K_SAMPLES.')

def build_case_text(row_data: Dict[str, Any]) -> str:
    parts: List[str] = []
    for column in CASE_COMPONENT_COLUMNS:
        value = row_data.get(column)
        if pd.isna(value):
            continue
        value_str = str(value).strip()
        if not value_str or value_str.lower() == 'nan':
            continue
        parts.append(f'{column}:\n{value_str}')
    return '\n\n'.join(parts)
df['case_text'] = df.apply(lambda row: build_case_text(row.to_dict()), axis=1)
print('Rows:', len(df))
print('Model:', SC_MODEL_NAME, '->', SC_MODEL_URI)
print('Provider:', SC_PROVIDER)
print('K_SAMPLES:', K_SAMPLES, '| K_LEVELS:', K_LEVELS)
print('Outputs:', list(OUTPUT_CSV_BY_K.values()))
total_calls = 0
print('\nСхема шага выбора:')
for k in K_LEVELS:
    n = usc_call_plan(k)
    total_calls += n * len(df)
    kind = 'полная орбита k!' if USC_DESIGN_BY_K[k] == 'orbit' else 'нокаут 2(k-1), пары в оба порядка'
    print(f'  SC@{k}: {USC_DESIGN_BY_K[k]:<9} — {kind:<34} {n:>2} вызовов/кейс')
print(f'Черновиков: {len(df)} x {K_SAMPLES} = {len(df) * K_SAMPLES:,}')
print(f'Вызовов USC: {total_calls:,}')
print(f'Бюджет Delphi для сравнения: {DELPHI_TOKENS_PER_CASE:,} токенов/кейс')


In [ ]:
session = requests.Session()
_api_semaphore = threading.Semaphore(MAX_INFLIGHT_CALLS)

def _backoff_seconds(attempt: int, error_text: str) -> float:
    delay = BASE_SLEEP_SECONDS * 2 ** attempt
    low = str(error_text).lower()
    if '429' in low or 'resourceexhausted' in low or 'quota' in low:
        delay *= 2
    return delay + random.uniform(0, 1.0)
llm_cache: Dict[str, Dict[str, Any]] = {}
openai_client = OpenAI(api_key=API_KEY, base_url='https://llm.api.cloud.yandex.net/v1', project=FOLDER_ID)

def make_cache_key(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, sample_idx: int=0) -> str:
    payload = {'provider': provider, 'model_uri': model_uri, 'system_text': system_text, 'user_text': user_text, 'temperature': temperature, 'max_tokens': max_tokens, 'sample_idx': sample_idx}
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _build_token_dict(prompt_tokens: int=0, completion_tokens: int=0, total_tokens: int=0) -> Dict[str, int]:
    return {'prompt_tokens': int(prompt_tokens or 0), 'completion_tokens': int(completion_tokens or 0), 'total_tokens': int(total_tokens or 0)}

def _cache_store(cache_key: str, text: str, usage: Dict[str, int]) -> None:
    llm_cache[cache_key] = {'text': text, **usage}

def _cache_load(cache_key: str) -> Dict[str, Any]:
    hit = llm_cache[cache_key]
    return {'text': hit['text'], 'prompt_tokens': int(hit.get('prompt_tokens', 0)), 'completion_tokens': int(hit.get('completion_tokens', 0)), 'total_tokens': int(hit.get('total_tokens', 0)), 'cached': True}

def _extract_openai_text(response: Any) -> str:
    if not getattr(response, 'choices', None):
        return ''
    message = response.choices[0].message
    content = getattr(message, 'content', '')
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                parts.append(item.get('text', ''))
            elif hasattr(item, 'type') and getattr(item, 'type', None) == 'text':
                parts.append(getattr(item, 'text', ''))
        return '\n'.join([p for p in parts if p]).strip()
    return ''

def _extract_openai_usage(usage: Any) -> Dict[str, int]:
    if usage is None:
        return _build_token_dict()
    prompt_tokens = getattr(usage, 'prompt_tokens', None)
    completion_tokens = getattr(usage, 'completion_tokens', None)
    total_tokens = getattr(usage, 'total_tokens', None)
    if total_tokens is None:
        input_tokens = getattr(usage, 'input_tokens', None)
        output_tokens = getattr(usage, 'output_tokens', None)
        prompt_tokens = prompt_tokens if prompt_tokens is not None else input_tokens
        completion_tokens = completion_tokens if completion_tokens is not None else output_tokens
        total_tokens = int(prompt_tokens or 0) + int(completion_tokens or 0)
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def _extract_yandex_usage(result: Dict[str, Any]) -> Dict[str, int]:
    usage = result.get('usage', {}) or {}
    prompt_tokens = usage.get('inputTextTokens', 0) or 0
    completion_tokens = usage.get('completionTokens', 0) or 0
    total_tokens = usage.get('totalTokens', 0) or prompt_tokens + completion_tokens
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def call_yandex_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, timeout: int=REQUEST_TIMEOUT, retries: int=MAX_RETRIES, sample_idx: int=0) -> Dict[str, Any]:
    cache_key = make_cache_key('yandex_rest', model_uri, system_text, user_text, temperature, max_tokens, sample_idx)
    if cache_key in llm_cache:
        return _cache_load(cache_key)
    headers = {'Content-Type': 'application/json', 'Authorization': f'Api-Key {API_KEY}'}
    payload = {'modelUri': model_uri, 'completionOptions': {'stream': False, 'temperature': temperature, 'maxTokens': max_tokens}, 'messages': [{'role': 'system', 'text': system_text}, {'role': 'user', 'text': user_text}]}
    last_error_text = ''
    for attempt in range(retries):
        try:
            with _api_semaphore:
                response = session.post(LLM_API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.RequestException as e:
            last_error_text = str(e)
            time.sleep(_backoff_seconds(attempt, last_error_text))
            continue
        if response.status_code == 200:
            data = response.json()
            result = data.get('result', {}) or {}
            raw_text = result.get('alternatives', [{}])[0].get('message', {}).get('text', '').strip()
            usage = _extract_yandex_usage(result)
            _cache_store(cache_key, raw_text, usage)
            return {'text': raw_text, **usage, 'cached': False}
        last_error_text = f'HTTP {response.status_code}: {response.text}'
        if response.status_code in (429, 500, 502, 503, 504):
            time.sleep(_backoff_seconds(attempt, last_error_text))
            continue
        raise RuntimeError(last_error_text)
    raise RuntimeError(f'Yandex REST call failed after {retries} retries. Last error: {last_error_text}')

def call_openai_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, retries: int=MAX_RETRIES, sample_idx: int=0) -> Dict[str, Any]:
    cache_key = make_cache_key('openai_compatible', model_uri, system_text, user_text, temperature, max_tokens, sample_idx)
    if cache_key in llm_cache:
        return _cache_load(cache_key)
    last_error_text = ''
    for attempt in range(retries):
        try:
            with _api_semaphore:
                response = openai_client.chat.completions.create(model=model_uri, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=temperature, max_tokens=max_tokens)
            raw_text = _extract_openai_text(response)
            usage = _extract_openai_usage(getattr(response, 'usage', None))
            _cache_store(cache_key, raw_text, usage)
            return {'text': raw_text, **usage, 'cached': False}
        except Exception as e:
            last_error_text = str(e)
            time.sleep(_backoff_seconds(attempt, last_error_text))
            continue
    raise RuntimeError(f'OpenAI-compatible call failed after {retries} retries. Last error: {last_error_text}')

def call_llm(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, sample_idx: int=0) -> Dict[str, Any]:

    def _dispatch() -> Dict[str, Any]:
        if provider == 'yandex_rest':
            return call_yandex_llm(model_uri, system_text, user_text, temperature, max_tokens, sample_idx=sample_idx)
        if provider == 'openai_compatible':
            return call_openai_llm(model_uri, system_text, user_text, temperature, max_tokens, sample_idx=sample_idx)
        raise ValueError(f'Unknown provider: {provider}')
    response = _dispatch()
    if str(response['text']).strip():
        return response
    llm_cache.pop(make_cache_key(provider, model_uri, system_text, user_text, temperature, max_tokens, sample_idx), None)
    retry = _dispatch()
    for key in ('prompt_tokens', 'completion_tokens', 'total_tokens'):
        retry[key] += response[key]
    if not str(retry['text']).strip():
        print(f'WARNING: empty response after retry (model_uri={model_uri}, max_tokens={max_tokens})')
    return retry


In [ ]:
def build_stage1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are a medical expert.\nAt this stage, you need to evaluate a specific clinical case and provide a final structured protocol based on the case.\n\nUse only information explicitly present in the case. If a detail is absent, write "Not specified"; if a field is unnecessary, write "None".\nReason carefully from the case. Do not invent symptoms or findings absent from the case.\n'.strip()

def build_stage1_user_prompt(case_text: str, author_id: str) -> str:
    emergency_list = json.dumps(EMERGENCY_LABELS)
    subclinical_list = json.dumps(SUBCLINICAL_LABELS)
    return f'Read the clinical case and write one medical protocol.\n\nClinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nWrite the protocol in plain text. Same field labels and order:\n\nAge:\nSex:\nComplaint:\nAnalysis:\nRequired diagnostic equipment/tests:\nConclusion:\nEmergency type:\nSubclinical specialization:\nRecommendations:\nNotes:\nProtocol Author: {author_id}\n\nField instructions:\nAge: copy from the case if stated; otherwise write "Not specified".\nSex: copy from the case if stated; otherwise write "Not specified".\nComplaint: briefly summarize the presenting problem from the case.\nAnalysis: explain the clinical reasoning, the strongest supporting facts, and the main relevant differential considerations.\nRequired diagnostic equipment/tests: mention only studies or equipment that are actually needed next; if nothing additional is needed, write "None".\nConclusion: give a concise clinical synthesis on the case.\nEmergency type: based on your analysis of this case, emergency type must equal exactly one of {emergency_list}.\nSubclinical specialization: based on your analysis of this case, subclinical specialization must equal exactly one of {subclinical_list}.\nRecommendations: provide practical, case-grounded next steps or management advice. Explicitly address the additional considerations.\nNotes: include only essential caveats; otherwise write "None".\n'.strip()

def strip_author_section(protocol_text: str) -> str:
    if protocol_text is None:
        return ''
    text = str(protocol_text).strip()
    parts = re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)
    return parts[0].rstrip()

def build_usc_system_prompt() -> str:
    return 'You are a careful expert assistant. You select among candidate responses. You never write a new response of your own.'.strip()

def build_usc_user_prompt(case_text: str, drafts: List[str]) -> str:
    blocks = '\n\n'.join((f'Response {i}:\n{d}' for i, d in enumerate(drafts, start=1)))
    return f'I have generated the following responses to the clinical case below.\n\nClinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\n{blocks}\n\nEvaluate these responses and select the most consistent response based on majority consensus.\nDo not write a new response. Do not merge or combine responses. Only select one of them.\n\nReturn plain text only, in exactly this format and nothing else:\nMost consistent response: <number>\n'.strip()

def parse_usc_choice(text: str, k: int) -> Dict[str, Any]:
    hits = re.findall('most\\s+consistent\\s+response\\s*:\\s*(\\d+)', str(text), flags=re.I)
    if hits:
        idx = int(hits[-1])
        if 1 <= idx <= k:
            return {'index': idx, 'parse_ok': True}
    return {'index': None, 'parse_ok': False}


In [ ]:
def _content_key(text: str) -> str:
    return hashlib.sha256(str(text).encode('utf-8')).hexdigest()

def _symmetric_pick(texts: List[str], label: str) -> int:
    ordered = sorted(range(len(texts)), key=lambda i: _content_key(texts[i]))
    digest = hashlib.sha256('|'.join([label] + [_content_key(texts[i]) for i in ordered]).encode('utf-8')).hexdigest()
    return ordered[int(digest[:16], 16) % len(ordered)]

def _usc_call(case_text: str, subset: List[str], order) -> Dict[str, Any]:
    shown = [subset[j] for j in order]
    resp = call_llm(provider=SC_PROVIDER, model_uri=SC_MODEL_URI, system_text=build_usc_system_prompt(), user_text=build_usc_user_prompt(case_text, shown), temperature=TEMPERATURE_USC, max_tokens=MAX_TOKENS_USC)
    choice = parse_usc_choice(resp['text'], len(order))
    return {'resp': resp, 'position': choice['index'] if choice['parse_ok'] else None, 'local': order[choice['index'] - 1] if choice['parse_ok'] else None}

def _select_orbit(case_text: str, subset: List[str], vignette: str) -> Dict[str, Any]:
    k = len(subset)
    orders = list(itertools.permutations(range(k)))
    with ThreadPoolExecutor(max_workers=min(len(orders), 6)) as pool:
        calls = list(pool.map(lambda o: _usc_call(case_text, subset, o), orders))
    votes = [c['local'] for c in calls if c['local'] is not None]
    counts = Counter(votes)
    if len(votes) < int(np.ceil(len(orders) * MIN_PARSED_CALL_RATE)):
        return {'winner_local': None, 'parse_ok': False, 'calls': calls, 'votes': counts, 'top': 0, 'n_tied': 0, 'vote_share': None, 'n_matches': None, 'n_tied_matches': None, 'tie_rate': None, 'bracket_path': ''}
    top = max(counts.values())
    tied = sorted((d for d, n in counts.items() if n == top))
    winner_local = tied[_symmetric_pick([subset[d] for d in tied], f'orbit|{vignette}|{k}')]
    return {'winner_local': winner_local, 'parse_ok': True, 'calls': calls, 'votes': counts, 'top': top, 'n_tied': len(tied), 'vote_share': top / len(votes), 'n_matches': None, 'n_tied_matches': None, 'tie_rate': None, 'bracket_path': ''}

def _match(case_text: str, subset: List[str], ia: int, ib: int, vignette: str):
    orders = [(ia, ib), (ib, ia)]
    with ThreadPoolExecutor(max_workers=2) as pool:
        calls = list(pool.map(lambda o: _usc_call(case_text, subset, o), orders))
    wins = Counter((c['local'] for c in calls if c['local'] is not None))
    if wins.get(ia, 0) != wins.get(ib, 0):
        return (ia if wins.get(ia, 0) > wins.get(ib, 0) else ib, False, calls)
    pick = _symmetric_pick([subset[ia], subset[ib]], f'match|{vignette}')
    return ((ia, ib)[pick], True, calls)

def _select_knockout(case_text: str, subset: List[str], vignette: str) -> Dict[str, Any]:
    k = len(subset)
    current = sorted(range(k), key=lambda i: _content_key(subset[i]))
    all_calls: List[Dict[str, Any]] = []
    path: List[str] = []
    had_bye: set = set()
    n_matches = n_tied_matches = 0
    rnd = 0
    while len(current) > 1:
        rnd += 1
        if len(current) % 2 == 1:
            cand = [i for i in reversed(current) if i not in had_bye]
            bye = cand[0] if cand else current[-1]
            rest = [i for i in current if i != bye]
            had_bye.add(bye)
        else:
            bye, rest = (None, current)
        pairs = [(rest[i], rest[i + 1]) for i in range(0, len(rest), 2)]
        with ThreadPoolExecutor(max_workers=min(len(pairs), 4)) as pool:
            results = list(pool.map(lambda p: _match(case_text, subset, p[0], p[1], vignette), pairs))
        nxt = []
        for (ia, ib), (winner, tied, calls) in zip(pairs, results):
            nxt.append(winner)
            all_calls.extend(calls)
            n_matches += 1
            n_tied_matches += int(tied)
            path.append(f"r{rnd}:{ia + 1}v{ib + 1}->{winner + 1}{('~' if tied else '')}")
        if bye is not None:
            nxt.append(bye)
            path.append(f'r{rnd}:bye{bye + 1}')
        current = nxt
    n_parsed = sum((1 for c in all_calls if c['local'] is not None))
    parse_ok = n_parsed >= int(np.ceil(len(all_calls) * MIN_PARSED_CALL_RATE))
    return {'winner_local': current[0] if parse_ok else None, 'parse_ok': parse_ok, 'calls': all_calls, 'votes': Counter(), 'top': 0, 'n_tied': 0, 'vote_share': None, 'n_matches': n_matches, 'n_tied_matches': n_tied_matches, 'tie_rate': n_tied_matches / n_matches if n_matches else None, 'bracket_path': ' '.join(path)}

def select_for_k(case_text: str, subset: List[str], vignette: str, design: str) -> Dict[str, Any]:
    if design == 'orbit':
        return _select_orbit(case_text, subset, vignette)
    if design == 'knockout':
        return _select_knockout(case_text, subset, vignette)
    raise ValueError(f'Unknown design: {design}')

def run_usc_case(row_data: Dict[str, Any]) -> Dict[str, Any]:
    case_text = row_data['case_text']
    vignette = row_data['Vignette']
    system_text = build_stage1_system_prompt(SC_ROLE_NAME, SC_ROLE_DESCRIPTION)
    user_text = build_stage1_user_prompt(case_text, SC_MODEL_NAME)

    def _draft(sample_idx: int) -> Dict[str, Any]:
        return call_llm(provider=SC_PROVIDER, model_uri=SC_MODEL_URI, system_text=system_text, user_text=user_text, temperature=TEMPERATURE_SC, max_tokens=MAX_TOKENS_PROTOCOL, sample_idx=sample_idx)
    with ThreadPoolExecutor(max_workers=K_SAMPLES) as pool:
        draft_responses = list(pool.map(_draft, range(K_SAMPLES)))
    drafts = [r['text'].strip() for r in draft_responses]
    for i, d in enumerate(drafts, start=1):
        row_data[f'sc_draft_{i}'] = d
    for i, r in enumerate(draft_responses, start=1):
        row_data[f'draft_{i}_total_tokens'] = int(r['total_tokens'])
    for k in K_LEVELS:
        subset = drafts[:k]
        design = USC_DESIGN_BY_K[k]
        res = select_for_k(case_text, subset, vignette, design)
        calls = res['calls']
        draft_index = res['winner_local'] + 1 if res['parse_ok'] else None
        winner = subset[res['winner_local']] if res['parse_ok'] else None
        d_prompt = sum((int(r['prompt_tokens']) for r in draft_responses[:k]))
        d_completion = sum((int(r['completion_tokens']) for r in draft_responses[:k]))
        d_total = sum((int(r['total_tokens']) for r in draft_responses[:k]))
        row_data[f'sc{k}_winner_protocol'] = winner
        row_data[f'sc{k}_usc_selected_index'] = draft_index
        row_data[f'sc{k}_usc_parse_ok'] = res['parse_ok']
        row_data[f'sc{k}_usc_design'] = design
        row_data[f'sc{k}_usc_n_calls'] = len(calls)
        row_data[f'sc{k}_usc_n_parsed'] = sum((1 for c in calls if c['local'] is not None))
        row_data[f'sc{k}_usc_votes'] = ','.join((f'{d + 1}:{n}' for d, n in sorted(res['votes'].items())))
        row_data[f'sc{k}_usc_top_votes'] = res['top']
        row_data[f'sc{k}_usc_n_tied'] = res['n_tied']
        row_data[f'sc{k}_usc_vote_share'] = res['vote_share']
        row_data[f'sc{k}_usc_n_matches'] = res['n_matches']
        row_data[f'sc{k}_usc_n_tied_matches'] = res['n_tied_matches']
        row_data[f'sc{k}_usc_tie_rate'] = res['tie_rate']
        row_data[f'sc{k}_usc_bracket_path'] = res['bracket_path']
        row_data[f'sc{k}_usc_shown_positions'] = ','.join((str(c['position']) for c in calls if c['position'] is not None))
        u_prompt = sum((int(c['resp']['prompt_tokens']) for c in calls))
        u_completion = sum((int(c['resp']['completion_tokens']) for c in calls))
        u_total = sum((int(c['resp']['total_tokens']) for c in calls))
        row_data[f'sc{k}_draft_total_tokens'] = d_total
        row_data[f'sc{k}_usc_prompt_tokens'] = u_prompt
        row_data[f'sc{k}_usc_completion_tokens'] = u_completion
        row_data[f'sc{k}_usc_total_tokens'] = u_total
        row_data[f'sc{k}_prompt_tokens'] = d_prompt + u_prompt
        row_data[f'sc{k}_completion_tokens'] = d_completion + u_completion
        row_data[f'sc{k}_total_tokens'] = d_total + u_total
        row_data[f'sc{k}_draft_budget_measured'] = d_total > 0
        row_data[f'sc{k}_usc_budget_measured'] = u_total > 0
        row_data[f'sc{k}_cached'] = all((bool(c['resp']['cached']) for c in calls)) and all((r['cached'] for r in draft_responses[:k]))
        row_data[f'sc{k}_usc_raw_response'] = ' ||| '.join((str(c['resp']['text']).replace('\n', ' ')[:120] for c in calls))
    return row_data

def run_usc_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:
    row_dicts = [row.to_dict() for _, row in input_df.iterrows()]
    with ThreadPoolExecutor(max_workers=CASE_CONCURRENCY) as pool:
        rows = list(tqdm(pool.map(run_usc_case, row_dicts), total=len(row_dicts), desc='SC@k + USC pipeline'))
    return pd.DataFrame(rows)
result_df = run_usc_pipeline(df)
print(result_df.shape)
for k in K_LEVELS:
    ok = result_df[f'sc{k}_usc_parse_ok'].astype(bool)
    print(f"\nSC@{k} [{USC_DESIGN_BY_K[k]}]: выбор состоялся {ok.sum()}/{len(ok)} ({ok.mean():.1%}), {result_df[f'sc{k}_usc_n_calls'].mean():.0f} вызовов/кейс, шаг USC {result_df[f'sc{k}_usc_total_tokens'].mean():,.0f} токенов/кейс, всего {result_df[f'sc{k}_total_tokens'].mean():,.0f} токенов/кейс")
    print('  выбранные черновики:', result_df[f'sc{k}_usc_selected_index'].value_counts(dropna=False).sort_index().to_dict())
print('\nСырое позиционное смещение в ответах модели (схемой не устраняется, ею сокращается):')
for k in K_LEVELS:
    pos = []
    for s in result_df[f'sc{k}_usc_shown_positions'].dropna():
        pos.extend((int(x) for x in str(s).split(',') if x))
    if not pos:
        continue
    width = 2 if USC_DESIGN_BY_K[k] == 'knockout' else k
    counts = pd.Series(pos).value_counts().reindex(range(1, width + 1), fill_value=0)
    print(f'  SC@{k}: {counts.tolist()} (n={len(pos)}), доля первой показанной = {counts.loc[1] / len(pos):.1%} (без смещения было бы {1 / width:.1%})')
print('\nСигнал шага выбора:')
rng = np.random.default_rng(0)
for k in K_LEVELS:
    if USC_DESIGN_BY_K[k] == 'orbit':
        share = result_df[f'sc{k}_usc_vote_share'].dropna().astype(float)
        if not len(share):
            continue
        n_calls = math.factorial(k)
        sim = rng.integers(0, k, size=(20000, n_calls))
        null = np.array([np.bincount(r, minlength=k).max() for r in sim]) / n_calls
        se = null.std() / np.sqrt(len(share))
        verdict = 'СИГНАЛА НЕТ' if share.mean() <= null.mean() + 2 * se else 'есть отклонение'
        print(f'  SC@{k} [orbit]: концентрация голосов {share.mean():.3f}, при случайном выборе {null.mean():.3f} (+/-{null.std():.3f})   {verdict}')
    else:
        tm = int(result_df[f'sc{k}_usc_n_tied_matches'].sum())
        mm = int(result_df[f'sc{k}_usc_n_matches'].sum())
        print(f'  SC@{k} [knockout]: ничьих 1:1 в {tm}/{mm} матчах ({tm / mm:.1%}); в {1 - tm / mm:.1%} матчей содержание пересилило позиционный гандикап')
result_df.head(3)


In [ ]:
N_INVARIANCE = 10
INVARIANCE_SEED = 20260826
tokens_before = sum((int(v.get('total_tokens', 0)) for v in llm_cache.values()))
rng_inv = random.Random(INVARIANCE_SEED)
rows = []
for _, row in result_df.head(N_INVARIANCE).iterrows():
    drafts = [str(row[c]) for c in DRAFT_COLUMNS]
    for k in K_LEVELS:
        subset = drafts[:k]
        perm = rng_inv.sample(range(k), k)
        renumbered = [subset[p] for p in perm]
        a = select_for_k(row['case_text'], subset, row['Vignette'], USC_DESIGN_BY_K[k])
        b = select_for_k(row['case_text'], renumbered, row['Vignette'], USC_DESIGN_BY_K[k])
        ta = subset[a['winner_local']] if a['parse_ok'] else None
        tb = renumbered[b['winner_local']] if b['parse_ok'] else None
        rows.append({'Vignette': row['Vignette'], 'k': k, 'design': USC_DESIGN_BY_K[k], 'perm': ''.join((str(p + 1) for p in perm)), 'match': ta is not None and ta == tb})
inv = pd.DataFrame(rows)
tokens_spent = sum((int(v.get('total_tokens', 0)) for v in llm_cache.values())) - tokens_before
print(f'Приёмочный тест инвариантности, {N_INVARIANCE} кейсов, seed={INVARIANCE_SEED}\n')
for k in K_LEVELS:
    g = inv[inv['k'] == k]
    print(f"  SC@{k} [{USC_DESIGN_BY_K[k]:<8}]: тот же черновик в {g['match'].sum()}/{len(g)} ({g['match'].mean():.0%})")
print(f'\nНовых токенов потрачено тестом: {tokens_spent:,} (ожидается 0 — при точной схеме перенумерация отправляет то же множество промптов)')
if not bool(inv['match'].all()):
    raise RuntimeError('ПРИЁМОЧНЫЙ ТЕСТ ПРОВАЛЕН: выбор зависит от нумерации черновиков. Проверить, что посев сетки, tie-break и выбор bye выведены из хешей ТЕКСТОВ (_content_key / _symmetric_pick), а не из индексов.')
print('\nТест пройден: выбор не зависит от нумерации черновиков.')
print('\nПроверка bye (нокаут):')
for k in K_LEVELS:
    if USC_DESIGN_BY_K[k] != 'knockout':
        continue
    dbl = 0
    for s in result_df[f'sc{k}_usc_bracket_path'].dropna():
        byes = [t for t in str(s).split() if 'bye' in t]
        ids = [int(t.split('bye')[1]) for t in byes]
        dbl += int(len(ids) != len(set(ids)))
    print(f"  SC@{k}: кейсов, где один участник получил bye дважды: {dbl}/{len(result_df)}{('  OK' if dbl == 0 else '  <-- ДЕФЕКТ')}")


In [ ]:
SERVICE_PREFIXES = tuple((f'sc{k}_' for k in K_LEVELS)) + ('draft_',)
MIN_PARSE_OK_RATE = 0.95
USC_DIAG_COLUMNS = ['usc_design', 'usc_selected_index', 'usc_parse_ok', 'usc_raw_response', 'usc_n_calls', 'usc_n_parsed', 'usc_votes', 'usc_top_votes', 'usc_n_tied', 'usc_vote_share', 'usc_n_matches', 'usc_n_tied_matches', 'usc_tie_rate', 'usc_bracket_path', 'usc_shown_positions', 'draft_total_tokens', 'draft_budget_measured', 'usc_budget_measured', 'usc_prompt_tokens', 'usc_completion_tokens', 'usc_total_tokens', 'prompt_tokens', 'completion_tokens', 'total_tokens', 'cached']

def build_arm(result_df: pd.DataFrame, k: int) -> pd.DataFrame:
    ok = result_df[f'sc{k}_usc_parse_ok'].astype(bool)
    dropped = int((~ok).sum())
    if dropped:
        print(f'SC@{k}: исключено {dropped} строк без разобранного выбора USC')
    keep = [c for c in result_df.columns if c not in DRAFT_COLUMNS and (not c.startswith(SERVICE_PREFIXES)) and (c not in set(USC_DIAG_COLUMNS) | {'sc_k', 'sc_temperature', 'sc_design', 'baseline_protocol', 'baseline_protocol_anonymous'})]
    arm = result_df.loc[ok, keep].copy()
    arm['baseline_protocol'] = result_df.loc[ok, f'sc{k}_winner_protocol']
    arm['baseline_protocol_anonymous'] = arm['baseline_protocol'].map(strip_author_section)
    arm['baseline_model_id'] = SC_MODEL_NAME
    arm['baseline_model_uri'] = SC_MODEL_URI
    arm['baseline_provider'] = SC_PROVIDER
    arm['baseline_role_name'] = SC_ROLE_NAME
    arm['baseline_role_description'] = SC_ROLE_DESCRIPTION
    arm['sc_k'] = k
    arm['sc_temperature'] = TEMPERATURE_SC
    arm['sc_design'] = USC_DESIGN_BY_K[k]
    for col in USC_DIAG_COLUMNS:
        arm[col] = result_df.loc[ok, f'sc{k}_{col}']
    return arm

def build_token_summary(arm: pd.DataFrame, k: int, n_dropped: int) -> pd.DataFrame:
    measured = arm['draft_budget_measured'].astype(bool) & arm['usc_budget_measured'].astype(bool)
    return pd.DataFrame([{'model_id': SC_MODEL_NAME, 'model_uri': SC_MODEL_URI, 'provider': SC_PROVIDER, 'role_name': SC_ROLE_NAME, 'sc_k': k, 'sc_design': USC_DESIGN_BY_K[k], 'sc_temperature': TEMPERATURE_SC, 'usc_calls_per_case': usc_call_plan(k), 'draft_total_tokens': int(arm['draft_total_tokens'].fillna(0).sum()), 'usc_total_tokens': int(arm['usc_total_tokens'].fillna(0).sum()), 'prompt_tokens': int(arm['prompt_tokens'].fillna(0).sum()), 'completion_tokens': int(arm['completion_tokens'].fillna(0).sum()), 'total_tokens': int(arm['total_tokens'].fillna(0).sum()), 'mean_usc_tokens_per_case': float(arm.loc[arm['usc_budget_measured'].astype(bool), 'usc_total_tokens'].mean()), 'mean_total_tokens_per_case': float(arm.loc[measured, 'total_tokens'].mean()), 'n_measured_rows': int(measured.sum()), 'n_cases': len(arm), 'n_cached': int(arm['cached'].fillna(False).sum()), 'n_usc_parse_fail': int(n_dropped)}])
arms = {k: build_arm(result_df, k) for k in K_LEVELS}
n_dropped = {k: int((~result_df[f'sc{k}_usc_parse_ok'].astype(bool)).sum()) for k in K_LEVELS}
summaries = {k: build_token_summary(arms[k], k, n_dropped[k]) for k in K_LEVELS}
parse_ok_rate = {k: float(result_df[f'sc{k}_usc_parse_ok'].astype(bool).mean()) for k in K_LEVELS}
print('\nДоля разобранных ответов USC:', ', '.join((f'SC@{k} {r:.1%}' for k, r in parse_ok_rate.items())))
for k in K_LEVELS:
    z = int((result_df[f'sc{k}_total_tokens'].fillna(0) == 0).sum())
    if z:
        print(f'ВНИМАНИЕ: SC@{k} — у {z} из {len(result_df)} строк расход нулевой; средний бюджет считается только по строкам с записанным usage.')
print('\nБюджет на кейс (сравнивается с Delphi):')
for k in K_LEVELS:
    s = summaries[k].iloc[0]
    print(f"  SC@{k} [{s['sc_design']:<8}] {int(s['usc_calls_per_case']):>2} выз.  {s['mean_total_tokens_per_case']:>8,.0f} ток./кейс  x{s['mean_total_tokens_per_case'] / DELPHI_TOKENS_PER_CASE:.2f} к Delphi")
pd.concat(list(summaries.values()), ignore_index=True)


In [ ]:
result_df.to_csv(RAW_OUTPUT_CSV, sep=';', index=False, encoding='utf-8-sig')
print('Saved:', RAW_OUTPUT_CSV, result_df.shape)
bad = {k: r for k, r in parse_ok_rate.items() if r < MIN_PARSE_OK_RATE}
if bad:
    raise RuntimeError(f'Плечи НЕ сохранены: доля разобранных ответов USC ниже порога {MIN_PARSE_OK_RATE:.0%} — ' + ', '.join((f'SC@{k} {r:.1%}' for k, r in bad.items())) + '. Черновики лежат в ' + RAW_OUTPUT_CSV + '; почините шаг USC и пересоберите плечи ремонтным ноутбуком.')
for k in K_LEVELS:
    arm_part = arms[k].copy()
    arm_part['table_source'] = 'result_df'
    summary_part = summaries[k].copy()
    summary_part['table_source'] = 'token_summary_df'
    combined_output_df = pd.concat([arm_part, summary_part], ignore_index=True, sort=False)
    out_csv = OUTPUT_CSV_BY_K[k]
    combined_output_df.to_csv(out_csv, sep=';', index=False, encoding='utf-8-sig')
    print('Saved:', out_csv, '| rows:', len(arm_part), '| total tokens:', summaries[k]['total_tokens'].iloc[0], '| mean/case:', round(summaries[k]['mean_total_tokens_per_case'].iloc[0], 1))
identical = (result_df[DRAFT_COLUMNS].nunique(axis=1) == 1).sum()
print(f'\nКейсов, где все {K_SAMPLES} черновиков идентичны: {identical} из {len(result_df)}' + ('  <-- ПРОВЕРИТЬ КЭШ И ТЕМПЕРАТУРУ' if identical else '  (ок)'))
